# PMAPS Workshop: IDAES-GTEP, Session 1

Welcome! In this tutorial, we will demonstrate how to get started with IDAES-GTEP using the PJM 5-bus test case as an example.

Additional IDEAS-GTEP resources:
- https://github.com/IDAES/idaes-gtep
- https://idaes-gtep.readthedocs.io/en/latest/index.html

Other tools leveraged by IDEAS-GTEP in this tutorial:
- Prescient: https://github.com/grid-parity-exchange/prescient
- EGRET: https://github.com/grid-parity-exchange/egret
- Pyomo: https://github.com/Pyomo/pyomo
- HiGHS: https://ergo-code.github.io/HiGHS/stable/

In [1]:
# Large amounts of warnings in this notebook are distracting; leaving this for now,
# unless we decide to resolve some of these warnings before the tutorial...

import logging
import warnings

logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

## Step 1: Reading in data

The `ExpansionPlanningData` class is our starting point. It allows us to define the temporal parameters of our model and reads in data defining the grid assets and how they are connected.

Its constructor takes the following arguments, all `int`:

| Name | Units | Default value | Description |
| --- | --- | --- | --- |
| `stages` | - | `2` | Number of investment periods |
| `num_reps` | - | `4` | Number of representative periods in each investment period |
| `len_reps` | Hours | `1` | Duration of each representative period |
| `num_commit` | - | `24` | Number of commitment periods in each representative period |
| `num_dispatch` | - | `1` | Number of dispatch periods in each commitment period |
| `duration_dispatch` | Minutes | `60` | Duration of each dispatch period |

Let's pick some values and create our `ExpansionPlanningData` object:

In [2]:
from gtep.gtep_data import ExpansionPlanningData

data_object = ExpansionPlanningData(
    stages=2,
    num_reps=2,
    len_reps=24,
    num_commit=2,
    num_dispatch=2,
)

Interactive Python mode detected; using default matplotlib backend for plotting.


Once the `ExpansionPlanningData` object is instantiated, we must point it to a directory containing data that will define our model setup. (DESCRIBE WHICH FILES ARE REQUIRED)

The IDAES-GTEP GitHub repository contains several example model setups, including one for the PJM 5-bus test case. Below, we navigate to this directory and list the data files it contains:

In [3]:
from pathlib import Path

data_path = (Path() / ".." / "data" / "5bus").resolve()
for fpath in data_path.iterdir():
    print(fpath)

C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\branch.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\bus.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\DAY_AHEAD_load.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\DAY_AHEAD_renewables.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\gen.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\initial_status.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\README.md
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\REAL_TIME_load.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\REAL_TIME_renewables.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\reserves.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\simulation_objects.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\timeseries_pointers.csv


Try opening `gen.csv` above. You should see 8 different generators, each with an associated bus and unit type/technology, among other attributes.

Once we have the path to our data directory, we pass it into the `load_prescient` method of our `ExpansionPlanningData` object, which uses the data loader from production cost modeling platform Prescient.

The following table summarizes the arguments for this method, which determine how the Prescient data loader is used:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `data_path` | `pathlib.Path` or `str` | - | Path to directory containing the data |
| `representative_dates` | `list[str]` | `None` (dates automatically chosen) | Representative dates to use; if provided, must have length `num_reps` |
| `representative_weights` | `list[float\|int]` | `None` (equal weights given) | Weight to give each representative date. If provided, must have length `num_reps` |
| `options_dict` | `dict` | `{"num_days": 365, "ruc_horizon": 36}` | Options passed to the Prescient data loader |

In [4]:
data_object.load_prescient(data_path)

Now that our data is loaded, it is stored under the `representative_data` attribute of the `ExpansionPlanningData` object. By printing its contents, we can see it consists of two EGRET `ModelData` objects, corresponding to the two representative periods that we defined in our `ExpansionPlanningData` constructor.

We can also explore the contents of these `ModelData` objects and see the data from our CSVs represented.

In [5]:
print(data_object.representative_data, "\n")

elements = data_object.representative_data[0].data["elements"]
print(elements.keys(), "\n")

for gen, gen_data in elements["generator"].items():
    print(f"{gen}    \t{gen_data['bus']}     \t{gen_data['generator_type']}    \t{gen_data['unit_type']}")

[<egret.data.model_data.ModelData object at 0x000001211E47CA50>, <egret.data.model_data.ModelData object at 0x000001211E44CD60>] 

dict_keys(['bus', 'load', 'shunt', 'area', 'branch', 'generator', 'storage']) 

3_CT    	bus3     	thermal    	CT
10_STEAM    	bus10     	thermal    	STEAM
4_CC    	bus4     	thermal    	CC
4_STEAM    	bus4     	thermal    	STEAM
10_PV    	bus10     	renewable    	PV
2_RTPV    	bus2     	renewable    	RTPV
1_HYDRO    	bus1     	renewable    	HYDRO
4_WIND    	bus4     	renewable    	WIND


## Step 2: Creating the model

The purpose of the `ExpansionPlanningModel` class is to build a Pyomo model for a grid expansion planning problem. The arguments of its constructor determine the physical setup of the problem (via the `ExpansionPlanningData` object from Step 1) as well as various cost and modeling options.

Specifically, the constructor takes the following arguments:
| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `data` | `ExpansionPlanningData` | - | Model data from Step 1 |
| `cost_data` | `DataProcessing` | `None` | Cost data. If not provided, placeholder values are assumed. For now we will leave this as `None`. To be explored in Session 2 |
| `config` | `dict` | `{}` | Model configuration options. For now, we will just use the default values. Alternative config options will be explored in Session 2 |
| `formulation` | - | `None` | Unused currently; to be implemented |

In [6]:
from gtep.gtep_model import ExpansionPlanningModel

mod_object = ExpansionPlanningModel(data=data_object)

Once the `ExpansionPlanningModel` object is created, we call its `create_model` method to build the corresponding Pyomo model.

In [7]:
mod_object.create_model()

[    0.00] Creating GTEP Model


The Pyomo model is now accessible from the `.model` attribute. Similarly, the `ExpansionPlanningData` object is accessible via `.model.data`, and the (first) EGRET `ModelData` object is accessible via `.model.md`.

In [8]:
print(type(mod_object.model))
print(type(mod_object.model.data))
print(type(mod_object.model.md))

<class 'pyomo.core.base.PyomoModel.ConcreteModel'>
<class 'gtep.gtep_data.ExpansionPlanningData'>
<class 'egret.data.model_data.ModelData'>


## Step 3. Solving the model

Once the Pyomo model has been created, it is up to the user to perform any transformations and solve the model using Pyomo utilities. At a minimum, we need to choose optimizer and pass in the Pyomo model. For a simple model like the one we defined, this is quick.

Note that Pyomo is compatible with several solvers, including some that require a license (e.g., Gurobi). Solvers aren't included in Pyomo or GTEP by default and must be installed into your Python environment manually. For the sake of this tutorial, we have already added HiGHS, a freely available solver compatible with Pyomo, to your environment (`highspy`).

In [9]:
from pyomo.environ import SolverFactory

opt = SolverFactory("highs")  # select your solver here
result = opt.solve(mod_object.model, tee=True)
for key, val in result.items():
    print(key, val)

Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
MIP has 234 rows; 546 cols; 1290 nonzeros; 120 integer variables (120 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+04]
  Cost    [1e+00, 1e+05]
  Bound   [1e+00, 1e+02]
  RHS     [2e+00, 5e+02]
Presolving model
34 rows, 146 cols, 130 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-234); columns 0(-546); nonzeros 0(-1290) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
Src  Proc. InQueue |  Lea

In some cases, it can be beneficial to perform transformations on the model before solving. The cell below compares the time to create, transform, and solve the model for two cases:
1. No transformations
2. Applying `gdp.bound_pretransformation` and `gdp.bigm`

With our simple model, the second case is slower, but transformations are useful to keep in mind for more complex models.

[TODO: Reference to available transformations?]

In [10]:
from pyomo.environ import TransformationFactory

def benchmark_model_times(data_object: ExpansionPlanningData):
    for transforms in [[], ["gdp.bound_pretransformation", "gdp.bigm"]]:

        mymodel = ExpansionPlanningModel(data=data_object)
        mymodel.create_model()
        mymodel.timer.toc("Created Model")

        for transform in transforms:
            TransformationFactory(transform).apply_to(mymodel.model)
            mymodel.timer.toc(f"Applied {transform}")

        opt.solve(mymodel.model)
        mymodel.timer.toc("Solved\n")

benchmark_model_times(data_object)

[    0.00] Creating GTEP Model
[+   0.72] Created Model
[+   0.19] Solved

[    0.00] Creating GTEP Model
[+   0.63] Created Model
[+   0.34] Applied gdp.bound_pretransformation
[+   0.67] Applied gdp.bigm
[+   1.47] Solved



## 4. Exploring results

The solution can be manually investigated by looking at the solved values for model variables.

For instance, we can investigate investment decisions, like which generators are operational at each investment stage `i`, by looking at:
- `investmentStage[i].genOperational[thermal_gen].indicator_var` for thermal generators (for which entire generator is either operational or not)
- `investmentStage[i].renewableOperational[thermal_gen]` for renewable generators (for which a numerical operational capacity is solved for)

See below.

In [83]:
import pyomo.environ as pyo

for i in mod_object.model.stages:
    print(f"Investment stage {i}")
    if i == mod_object.model.stages.last():
        print(f"[Skipping thermal generators status for investment stage {i}] -- WHY???")
    else:
        for thermal_gen in mod_object.model.thermalGenerators:
            print(thermal_gen, "   \t", pyo.value(mod_object.model.investmentStage[i].genOperational[thermal_gen].indicator_var))

    for renewable_gen in mod_object.model.renewableGenerators:
        print(renewable_gen, "   \t", pyo.value(mod_object.model.investmentStage[i].renewableOperational[renewable_gen]))
    print()

Investment stage 1
3_CT    	 True
10_STEAM    	 True
4_CC    	 True
4_STEAM    	 True
10_PV    	 20.7
2_RTPV    	 7.8
1_HYDRO    	 39.117
4_WIND    	 118.486

Investment stage 2
[Skipping thermal generators status for investment stage 2] -- WHY???
10_PV    	 20.7
2_RTPV    	 7.8
1_HYDRO    	 39.117
4_WIND    	 118.486



We can also look at dispatch-level variables, for instance the amount of power generated by each generator, by accessing the `thermalGeneration` and `renewableGeneration` variables on each dispatch block.

Below, we pull out a dispatch block from each representative period:

In [84]:
dispatch_blocks = {
    r: mod_object.model
        .investmentStage[1]
        .representativePeriod[r]
        .commitmentPeriod[1]
        .dispatchPeriod[1]
    for r in mod_object.model.representativePeriods
}

for r, b in dispatch_blocks.items():
    print(f"Representative period {r}")
    for key, val in b.thermalGeneration.items():
        print(key, "   \t", pyo.value(val))
    for key, val in b.renewableGeneration.items():
            print(key, "   \t", pyo.value(val))
    print()

Representative period 1
3_CT    	 20.0
10_STEAM    	 76.0
4_CC    	 100.0
4_STEAM    	 12.0
10_PV    	 0.0
2_RTPV    	 0.0
1_HYDRO    	 13.658
4_WIND    	 118.486

Representative period 2
3_CT    	 20.0
10_STEAM    	 76.0
4_CC    	 100.0
4_STEAM    	 12.0
10_PV    	 0.0
2_RTPV    	 0.0
1_HYDRO    	 10.592
4_WIND    	 2.186



To streamline extracting this data, we can leverage the `ExpansionPlanningSolution` class. Its constructor takes a single required argument argument (the path to the data we used to construct the model).

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `data_path` | `Path` or `str` | - | Path to model data from Step 1 |

In [85]:
from soraya_solution import ExpansionPlanningSolution

soln = ExpansionPlanningSolution(data_path)

We can then call the class's `save_results_in_json_files` method to automatically write out data from the model solution. It takes two arguments:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `gtep_model` | `ExpansionPlanningModel` | - | Model object we solved in Step 3 |
| `soln_path` | `Path` or `str` | - | Path to write solution files to |

In [86]:
soln_path = (Path() / "soln").resolve()
soln.save_results_in_json_files(mod_object, soln_path)

ERROR: evaluating object as numeric value:
investmentStage[1].genDisabled[3_CT].indicator_var
        (object: <class 'pyomo.gdp.disjunct.AutoLinkedBooleanVar'>)
    No value for uninitialized AutoLinkedBooleanVar object
    investmentStage[1].genDisabled[3_CT].indicator_var
Woops: No value for uninitialized AutoLinkedBooleanVar object investmentStage[1].genDisabled[3_CT].indicator_var
ERROR: evaluating object as numeric value:
investmentStage[1].genDisabled[10_STEAM].indicator_var
        (object: <class 'pyomo.gdp.disjunct.AutoLinkedBooleanVar'>)
    No value for uninitialized AutoLinkedBooleanVar object
    investmentStage[1].genDisabled[10_STEAM].indicator_var
Woops: No value for uninitialized AutoLinkedBooleanVar object investmentStage[1].genDisabled[10_STEAM].indicator_var
ERROR: evaluating object as numeric value:
investmentStage[1].genDisabled[4_CC].indicator_var
        (object: <class 'pyomo.gdp.disjunct.AutoLinkedBooleanVar'>)
    No value for uninitialized AutoLinkedBoolean

Finally, the `ExpansionPlanningSolution` class has a `create_plots` method, which produces several interactive html plots to help understand investment decisions made by the model:

In [87]:
soln.create_plots("renewables", soln_path, data_path)

 Created the subdirectory 'C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\tutorials\soln\plots' to save the plots.
Candidate generators file not found; skipping.
Storage csv not present; skipping
 -> Saved interactive stack plot for generation mix to C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\tutorials\soln/plots/investment_renewables_gen_mix_summary_interactive.html
 -> Saved interactive treemap for 2020 to C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\tutorials\soln/plots/renewables_treemap_2020_interactive.html
 -> Saved interactive pie chart for 2020 to C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\tutorials\soln/plots/renewables_pie_leader_2020_interactive.html


## Using GTEP to explore the PJM 5-bus test case

Ideas from Kyle on what to try out:
- what happens if load dramatically increases (e.g. datacenter)
- what does that look like over a 5-year period, or iterating over several periods
- what if we can only build batteries, what's the cheapest way to meet demand?
- etc.

Want to get a better understanding of what is actually at the different buses. Trying to print some info from the data object:

In [106]:
for bus_name, bus_data in data_object.md.elements("bus"):
    print(bus_name, bus_data)
    for gen_name, gen_data in data_object.md.elements("generator"):
        if gen_data["bus"] == bus_name:
            print(gen_name)
    for load_name, load_data in data_object.md.elements("load"):
        if load_name == bus_name:
            print("LOAD")
    print()

bus1 {'id': '1', 'base_kv': 230.0, 'matpower_bustype': 'PV', 'vm': 1.0, 'va': 0.048935018, 'v_min': 0.95, 'v_max': 1.05, 'area': '1', 'zone': '1'}
1_HYDRO

bus4 {'id': '4', 'base_kv': 230.0, 'matpower_bustype': 'ref', 'vm': 1.0, 'va': 0.0, 'v_min': 0.95, 'v_max': 1.05, 'area': '1', 'zone': '1'}
4_CC
4_STEAM
4_WIND
LOAD

bus10 {'id': '10', 'base_kv': 230.0, 'matpower_bustype': 'PV', 'vm': 1.0, 'va': 0.06266308, 'v_min': 0.95, 'v_max': 1.05, 'area': '1', 'zone': '1'}
10_STEAM
10_PV

bus2 {'id': '2', 'base_kv': 230.0, 'matpower_bustype': 'PQ', 'vm': 1.04407, 'va': -0.012822061, 'v_min': 0.95, 'v_max': 1.05, 'area': '2', 'zone': '1'}
2_RTPV
LOAD

bus3 {'id': '3', 'base_kv': 230.0, 'matpower_bustype': 'PV', 'vm': 1.0, 'va': -0.009768957, 'v_min': 0.95, 'v_max': 1.05, 'area': '2', 'zone': '1'}
3_CT
LOAD

